# sorted-computational-graph — ex1: topological sort of the compute graph for the reverse pass

> Procedural drill from [Delta Drills](https://delta-drills.vercel.app).
> Atom: `sorted-computational-graph`. Running the final beacon cell reports progress against the `Backprop: Sorted computation graph` subtopic.

**Why this is a Colab exercise.** This standalone exercises material the Delta Drills flashcards can't deliver on their own — interactive tensor execution, visualization, or multi-step debugging. Read the prompt, fill in the function body, run the test cell, then run the beacon at the bottom.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)
# === manual autograd primitives — shared across all drills in this folder ===
from dataclasses import dataclass, field
from typing import Any, Callable, Optional

grad_tracking_enabled = True

@dataclass
class Recipe:
    func: Optional[Callable] = None
    args: tuple = ()
    kwargs: dict = field(default_factory=dict)
    parents: dict = field(default_factory=dict)

class MiniTensor:
    """A minimal Tensor wrapper for the ARENA-style manual-autograd drills.
    Wraps a raw `torch.Tensor` in `.array`. Carries an optional `.recipe`
    populated by wrap_forward_fn. `requires_grad` is set by the wrapper."""
    def __init__(self, array, requires_grad: bool = False, recipe=None):
        self.array = array
        self.requires_grad = requires_grad
        self.recipe = recipe
    def __repr__(self):
        return f'MiniTensor({self.array!r}, requires_grad={self.requires_grad})'

## Connect to Delta Drills

Paste your Delta Drills auth token below so this drill can report progress on the `Backprop: Sorted computation graph` subtopic. Copy it from your Delta Drills account page.

This standalone exercises the atom **`sorted-computational-graph`** (exercise 1). Completion fires the beacon at the bottom.

In [ ]:
# === Delta Drills auth ===
DD_TOKEN = ""  # paste your token here, then run this cell
DD_ATOM_ID = "sorted-computational-graph"
DD_SUBTOPIC = "Backprop: Sorted computation graph"
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

_dd_passed = set()

## Sorted computational graph — quick refresher

The reverse pass needs the nodes in an order such that **every node comes BEFORE its parents**, so by the time we pop a node off the iteration, all the gradients flowing into it have already been accumulated.

Recipe (assumes you already have `topological_sort(node, get_children)` that returns descendants of `node` in DAG order where `node` is LAST):

```python
def sorted_computational_graph(tensor: MiniTensor) -> list[MiniTensor]:
    def get_parents(t):
        if t.recipe is None:
            return []
        return list(t.recipe.parents.values())
    return topological_sort(tensor, get_parents)[::-1]
```

Two key choices:
- **`get_parents` returns `[]` for leaves.** A leaf has no `recipe`; the traversal stops there.
- **Reverse the result.** `topological_sort` ends with the root (the end node) LAST; the reverse pass wants the root FIRST so it can seed the grad accumulator and walk backward. Reversal flips the order — `[::-1]` is the one-liner.

After reversal, `result[0] is end_node`, `result[-1]` is some leaf, and iterating in order means every node's gradient has been summed by the time the dispatcher needs it.

### Exercise 1 — topological sort of the compute graph for the reverse pass

> ```yaml
> Difficulty: 🔴🔴🔴🔴⚪
> Bloom level: Apply
> LO: Apply topological sort over a MiniTensor's recipe-parents graph and reverse the result so the end node comes first — the order the reverse pass needs.
> Keywords: topological-sort, reverse-order, get-parents, recipe-walk
> ```

**KCs targeted:** `sorted-computational-graph`, `parents-dict-by-argidx`

Implement TWO pieces:

**1. `topological_sort(node, get_children)`** — generic DFS-based topological sort. Returns descendants of `node` such that `node` is LAST (every parent appears before its children's dependencies). **Must raise `ValueError` on a cycle** (we're a DAG-only system; circular Recipes mean somebody's mutating during forward).

Classic three-color DFS:
- `temp` (gray) — currently on the DFS stack. Re-visiting one of these means we found a back-edge → cycle.
- `perm` (black) — fully processed; skip.
- Anything else (white) — not yet visited; recurse into.

**2. `sorted_computational_graph(tensor)`** — apply `topological_sort` over MiniTensor's parent graph and **reverse** the result. After this call:
- `result[0] is tensor` (the end node — where the reverse pass starts).
- `result[-1]` is some leaf MiniTensor (`.recipe is None`).
- Iterating in order means every node's accumulated gradient is fully summed by the time the dispatcher needs it.

Use `get_parents(t) = list(t.recipe.parents.values())` if `t.recipe is not None`, else `[]`. Then `return topological_sort(tensor, get_parents)[::-1]`.

**Why reverse.** `topological_sort` is designed for forward graphs (deps first). The reverse pass needs nodes with **outgoing edges resolved first** — i.e. start at the root, walk to leaves — which is the same DAG in reverse order. The `[::-1]` is the cheapest way to flip orientation.

Helpers (`Recipe`, `MiniTensor`) are in the setup cell already.

In [ ]:
def topological_sort(node, get_children):
    result = []
    perm = set()    # fully processed nodes (black) — by id() since MiniTensor isn't hashable-by-equality
    temp = set()    # currently on the DFS stack (gray) — cycle detector

    def visit(cur):
        cid = id(cur)
        if cid in perm:
            return
        if cid in temp:
            raise ValueError(f'Cycle detected at node {cur!r} — graph is not a DAG')
        temp.add(cid)
        for child in get_children(cur):
            visit(child)
        temp.remove(cid)
        perm.add(cid)
        result.append(cur)

    visit(node)
    return result


def sorted_computational_graph(tensor):
    def get_parents(t):
        if t.recipe is None:
            return []
        return list(t.recipe.parents.values())
    # topological_sort returns deps-first (end node LAST); reverse for the
    # backward pass which wants end node FIRST.
    return topological_sort(tensor, get_parents)[::-1]


<details><summary>Solution</summary>

```python
def topological_sort(node, get_children):
    result = []
    perm = set()    # fully processed nodes (black) — by id() since MiniTensor isn't hashable-by-equality
    temp = set()    # currently on the DFS stack (gray) — cycle detector

    def visit(cur):
        cid = id(cur)
        if cid in perm:
            return
        if cid in temp:
            raise ValueError(f'Cycle detected at node {cur!r} — graph is not a DAG')
        temp.add(cid)
        for child in get_children(cur):
            visit(child)
        temp.remove(cid)
        perm.add(cid)
        result.append(cur)

    visit(node)
    return result


def sorted_computational_graph(tensor):
    def get_parents(t):
        if t.recipe is None:
            return []
        return list(t.recipe.parents.values())
    # topological_sort returns deps-first (end node LAST); reverse for the
    # backward pass which wants end node FIRST.
    return topological_sort(tensor, get_parents)[::-1]
```

**Why `id(...)` instead of the node itself for set membership.** MiniTensors compare by identity already (we didn't override `__eq__` or `__hash__`), so `set` would work — but for objects that DO have value-equality (numpy arrays, torch tensors with custom `__eq__`) you'd get false positives. `id(...)` is the safe-by-default identity key.

**Why three colors, not two.** Two colors (visited / unvisited) catches re-visits but doesn't distinguish 'I've finished this subtree' from 'I'm in the middle of this subtree' — i.e. it can't detect cycles. The `temp`/`perm` split is the standard DFS topo-sort idiom: temp catches back-edges (cycles), perm avoids re-processing shared descendants in branching DAGs.

**Why `[::-1]` at the end of `sorted_computational_graph`.** `topological_sort` is generic — it's also useful for forward operations (deps-first). The reverse pass needs the OPPOSITE order: end node first, leaves last. Reversing keeps the generic sort reusable. Alternative: write a `reverse_topological_sort` that builds the list in reverse order natively (slightly faster, no reversal cost; we choose clarity here).
</details>

## Report completion

Run the cell below to send your progress to Delta Drills. The beacon fires only if the test cell above passed.

In [ ]:
# === Delta Drills completion beacon ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'ex1'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'procedural-drill:{DD_ATOM_ID}:ex1',
        'subtopics': [DD_SUBTOPIC],
        'feedback': 'somewhat',
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported {DD_ATOM_ID} (subtopic={DD_SUBTOPIC!r})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()